In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Howard et al. (JCP 493, 2023, 112462)
NON-COMPOSITE MF DEEPONET | VERSION-A | GP SCALE s=0.35
======================================================================

Purpose
-------
Run the Layer-1 canonical non-composite multifidelity DeepONet on the
unique Version-A s=0.35 benchmark. This file is intended for the final
Raw-LF / non-composite / ICL comparison.

This script is intentionally kept as close as possible to the earlier
composite MF script so the two runs can be compared cleanly.

What stays the same as the composite MF run
-------------------------------------------
* Same dataset and t=50 train/test split.
* Same 20 paired HF training cases and same 50 held-out test cases.
* Same HF input: (beta,H) on the 41x41 grid.
* Same linear correlation DeepONet: 1 layer, latent width 10, no activation.
* Same nonlinear modified DeepONet: 5 layers, width/latent 200, tanh also on
  its last layer.
* Same vector-output split: first half latent features for u, second half for v.
* Same JAX initialization style.
* Same seed = 20260827.
* The shared linear/nonlinear subnet initialization keys are deliberately
  chosen to be the SAME subkeys used in the composite code.
* Same Adam schedule: (1e-4, 5000, 0.97).
* Same HF batch size = 5.
* Same 600,000 training steps.
* Same input nondimensionalization beta/3000, H/1000, coords -> [-1,1].
* Same physical-unit velocity loss (no output standardization).
* Same final-fixed-budget official evaluation.
* Same HF minibatch RNG sequence as the composite code: we deliberately
  consume the unused LF minibatch draw each step before drawing the HF batch.

What changes -- exactly the Appendix-D idea
-------------------------------------------
The composite model first learns

    (beta,H) -> Uc_hat

with a low-fidelity DeepONet, then feeds Uc_hat to the correlation networks.

The non-composite model removes that LF subnet and directly supplies the
AVAILABLE paired low-fidelity solution Uc to the correlation networks.
Appendix D, Fig. D.24 / Eq. (D.1), uses:

    nonlinear branch input = [u(x_1..x_MH), yL(u)(x_1..x_QH)]
    linear branch input    =  yL(u)(x_1..x_PH)

For this 2D vector-valued benchmark we use the same dimensions as our
composite run:

    M_H = 2*41^2 = 3362
    P_H = 2*41^2 = 3362
    Q_H = P_L = 2*15^2 = 450

Thus:
* nonlinear branch receives normalized HF (beta,H), plus the ACTUAL native
  15x15 Uc (flattened to 450 scalar values);
* linear branch receives the SAME ACTUAL Uc bilinearly evaluated on the
  41x41 HF output grid (3362 scalar values).

The 15x15 -> 41x41 interpolation uses the physical x/y grids.  This is the
same align-the-grid bilinear representation used for Uc in our ICL protocol.

Non-composite loss
------------------
Howard Appendix D Eq. (D.1):

    L_noncomp = lambda1 * L_HF + lambda3 * R_NL.

There is no LF data-fit term and no LF-subnetwork regularizer because the LF
subnetwork no longer exists.  The linear subnet is unregularized, matching
Eq. (D.1).

Final hyperparameter convention
-------------------------------
For this controlled ice-sheet adaptation we retain the Sec. 3.4.2/Table C.7
weights that remain in the Appendix-D non-composite objective:

    lambda_HF = 1
    lambda_NLreg = 1e-3

This is our fixed Layer-1 convention; no alternate weight variant is run here.

Dataset
-------
Expected Kaggle root:

    /kaggle/input/notebooks/xinyuanli913/dataset-ice-sheet-final

Expected file somewhere recursively under it:

    howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz

The generator pairs:
    LF train[0:20] <-> HF train[0:20]
and provides 50 paired LF/HF held-out test cases.

This is a controlled adaptation of Howard Appendix D to our independently
generated benchmark, NOT an author-reported non-composite ice-sheet result.
"""

from __future__ import annotations

import csv
import json
import hashlib
import os
import math
import pickle
import time
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
from scipy.interpolate import RegularGridInterpolator

import jax
import jax.numpy as jnp
from jax import jit, value_and_grad
from jax.example_libraries import optimizers


# =============================================================================
# CONFIG -- SAME VERSION-A DATASET AS THE COMPOSITE RUN
# =============================================================================

INPUT_ROOT = os.environ.get("LAYER1_INPUT_ROOT", "/kaggle/input")
DATASET_PATH_OVERRIDE = os.environ.get("LAYER1_DATASET_PATH", "").strip()
VERSION_A_FILENAME = "howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz"

SAVE_ROOT = os.environ.get("LAYER1_NONCOMP_SAVE_ROOT", "/kaggle/working/LAYER1_NONCOMPOSITE_FINAL")

SEED = 20260827
TRAIN_STEPS = 600_000

# Keep this draw size only so we can consume exactly the same RNG call that
# the composite code used before drawing its HF minibatch.
LF_BATCH_FOR_RNG_MATCH = 10
HF_BATCH = 5

LOG_EVERY = 1_000
CHECKPOINT_EVERY = 100_000
EVAL_BATCH = 5

HIDDEN = 200
LATENT = 200
N_LAYERS = 5
LINEAR_LATENT = 10

LR0 = 1.0e-4
LR_DECAY_STEPS = 5_000
LR_DECAY_RATE = 0.97

# Final Layer-1 convention: Appendix-D non-composite loss with the
# Sec. 3.4.2/Table C.7 coefficients that remain after removing the LF subnet.
LAMBDA_SOURCE = "layer1_final_table_c7"
LAMBDA_HF = 1.0
LAMBDA_NL_REG = 1.0e-3

# Fixed network-input nondimensionalization.  Uc enters in physical m/yr,
# exactly as learned Uc_hat did in the composite implementation.
BETA_SCALE = 3000.0
H_SCALE = 1000.0



# =============================================================================
# DATA PATH
# =============================================================================

def resolve_data_path(filename: str) -> Path:
    if DATASET_PATH_OVERRIDE:
        p = Path(DATASET_PATH_OVERRIDE)
        if not p.exists():
            raise FileNotFoundError(f"LAYER1_DATASET_PATH does not exist: {p}")
        if p.name != filename:
            raise ValueError(f"Dataset override must point to {filename}, got {p.name}")
        print(f"[data-path override] {p}")
        return p

    root = Path(INPUT_ROOT)
    if not root.exists():
        raise FileNotFoundError(
            f"Kaggle input root not found:\n  {root}\n"
            "Attach the canonical dataset notebook output as a Kaggle input."
        )

    matches = sorted(root.rglob(filename))
    if len(matches) == 1:
        print(f"[data-path] {filename}\n  -> {matches[0]}")
        return matches[0]

    if len(matches) == 0:
        npz = sorted(root.rglob("*.npz"))
        preview = "\n".join(f"  - {p}" for p in npz[:80]) or "  (none)"
        raise FileNotFoundError(
            f"Could not find:\n  {filename}\n"
            f"under:\n  {root}\n\nVisible NPZ files:\n{preview}"
        )

    raise RuntimeError(
        f"Multiple copies of {filename} were found; refusing to choose silently:\n"
        + "\n".join(f"  - {p}" for p in matches)
    )


# =============================================================================
# REPRODUCIBILITY / BASIC HELPERS
# =============================================================================

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def audit_canonical_metadata(d: Dict[str, np.ndarray]) -> Dict[str, Any]:
    if "metadata_json" not in d:
        raise KeyError("Canonical dataset must contain metadata_json")
    raw = d["metadata_json"]
    if isinstance(raw, np.ndarray):
        raw = raw.item()
    meta = json.loads(str(raw))
    scale = meta.get("beta_gp_scale")
    if scale is None and isinstance(meta.get("config"), dict):
        scale = meta["config"].get("beta_gp_scale")
    if scale is None:
        raise KeyError("metadata_json does not contain beta_gp_scale")
    if abs(float(scale) - 0.35) > 1e-12:
        raise ValueError(f"Wrong Layer-1 dataset: beta_gp_scale={scale}, expected 0.35")
    return meta


def make_rng(seed: int) -> np.random.Generator:
    return np.random.default_rng(seed)


def flatten_input(beta: np.ndarray, H: np.ndarray) -> np.ndarray:
    """[N,ny,nx] beta,H -> [N,2*ny*nx], same as composite code."""
    beta_n = np.asarray(beta, dtype=np.float32) / np.float32(BETA_SCALE)
    H_n = np.asarray(H, dtype=np.float32) / np.float32(H_SCALE)
    return np.concatenate(
        [beta_n.reshape(len(beta_n), -1), H_n.reshape(len(H_n), -1)],
        axis=1,
    ).astype(np.float32)


def make_coords(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Map trunk coordinates to [-1,1]^2, same as composite code."""
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    xmin, xmax = float(x.min()), float(x.max())
    ymin, ymax = float(y.min()), float(y.max())

    xn = 2.0 * (x - xmin) / max(xmax - xmin, 1.0e-12) - 1.0
    yn = 2.0 * (y - ymin) / max(ymax - ymin, 1.0e-12) - 1.0
    X, Y = np.meshgrid(xn, yn, indexing="xy")
    return np.stack([X.ravel(), Y.ravel()], axis=1).astype(np.float32)


def resample_scalar_batch(
    fields: np.ndarray,
    x_src: np.ndarray,
    y_src: np.ndarray,
    x_dst: np.ndarray,
    y_dst: np.ndarray,
) -> np.ndarray:
    """Bilinear interpolation of [N,ny,nx] fields on regular physical grids."""
    fields = np.asarray(fields, dtype=np.float32)
    x_src = np.asarray(x_src, dtype=np.float64)
    y_src = np.asarray(y_src, dtype=np.float64)
    x_dst = np.asarray(x_dst, dtype=np.float64)
    y_dst = np.asarray(y_dst, dtype=np.float64)

    Xd, Yd = np.meshgrid(x_dst, y_dst, indexing="xy")
    pts = np.column_stack([Yd.ravel(), Xd.ravel()])

    out = np.empty((len(fields), len(y_dst), len(x_dst)), dtype=np.float32)
    for i in range(len(fields)):
        f = RegularGridInterpolator(
            (y_src, x_src),
            fields[i].astype(np.float64),
            method="linear",
            bounds_error=True,
        )
        out[i] = f(pts).reshape(len(y_dst), len(x_dst)).astype(np.float32)
    return out


def resample_velocity_batch(
    U: np.ndarray,
    x_src: np.ndarray,
    y_src: np.ndarray,
    x_dst: np.ndarray,
    y_dst: np.ndarray,
) -> np.ndarray:
    """
    Bilinear interpolation of vector velocity [N,2,ny,nx].

    Because x_src/x_dst and y_src/y_dst span the same endpoints, this is the
    physical-grid equivalent of bilinear interpolation with align_corners=True,
    matching the Uc representation used in the ICL Version-A protocol.
    """
    U = np.asarray(U, dtype=np.float32)
    out = np.empty((len(U), 2, len(y_dst), len(x_dst)), dtype=np.float32)
    for c in range(2):
        out[:, c] = resample_scalar_batch(U[:, c], x_src, y_src, x_dst, y_dst)
    return out


def per_case_rel_l2(pred: np.ndarray, truth: np.ndarray) -> np.ndarray:
    pred = np.asarray(pred, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    num = np.linalg.norm((pred - truth).reshape(len(pred), -1), axis=1)
    den = np.linalg.norm(truth.reshape(len(truth), -1), axis=1) + 1.0e-12
    return num / den


def mean_mse(pred: np.ndarray, truth: np.ndarray) -> float:
    pred = np.asarray(pred, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    return float(np.mean((pred - truth) ** 2))


def summarize(pred: np.ndarray, truth: np.ndarray) -> Dict[str, float]:
    r = per_case_rel_l2(pred, truth)
    return {
        "mean_rel_l2": float(r.mean()),
        "std_rel_l2": float(r.std()),
        "median_rel_l2": float(np.median(r)),
        "p90_rel_l2": float(np.quantile(r, 0.90)),
        "max_rel_l2": float(r.max()),
        "mean_mse": mean_mse(pred, truth),
    }


def fmt_stats(s: Dict[str, float]) -> str:
    return (
        f"RelL2 mean={100*s['mean_rel_l2']:.4f}% | "
        f"std={100*s['std_rel_l2']:.4f}% | "
        f"median={100*s['median_rel_l2']:.4f}% | "
        f"p90={100*s['p90_rel_l2']:.4f}% | "
        f"max={100*s['max_rel_l2']:.4f}% | "
        f"MSE={s['mean_mse']:.6f}"
    )


# =============================================================================
# LOAD VERSION-A MASTER
# =============================================================================

def load_version_a(path: Path) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)

    required = [
        "x_lf", "y_lf", "x_hf", "y_hf",
        "beta_lf_train", "H_lf_train", "U_lf_train",
        "beta_hf_train", "H_hf_train", "U_hf_train",
        "beta_lf_test", "H_lf_test", "U_lf_test",
        "beta_hf_test", "H_hf_test", "U_hf_test",
    ]
    missing = [k for k in required if k not in z.files]
    if missing:
        raise KeyError(f"Version-A dataset is missing required arrays: {missing}")

    d = {k: z[k] for k in z.files}

    assert d["beta_lf_train"].shape == (100, 15, 15)
    assert d["H_lf_train"].shape == (100, 15, 15)
    assert d["U_lf_train"].shape == (100, 2, 15, 15)

    assert d["beta_hf_train"].shape[1:] == (41, 41) and d["beta_hf_train"].shape[0] >= 20
    assert d["H_hf_train"].shape == d["beta_hf_train"].shape
    assert d["U_hf_train"].shape == (d["beta_hf_train"].shape[0], 2, 41, 41)

    assert d["beta_lf_test"].shape == (50, 15, 15)
    assert d["H_lf_test"].shape == (50, 15, 15)
    assert d["U_lf_test"].shape == (50, 2, 15, 15)

    assert d["beta_hf_test"].shape == (50, 41, 41)
    assert d["H_hf_test"].shape == (50, 41, 41)
    assert d["U_hf_test"].shape == (50, 2, 41, 41)

    return d


# =============================================================================
# JAX PARAMETER INITIALIZATION -- SHARED BLOCKS MATCH COMPOSITE SUBKEYS
# =============================================================================

glorot_normal = jax.nn.initializers.glorot_normal()
zeros_init = jax.nn.initializers.zeros


def init_dense(key: jax.Array, din: int, dout: int) -> Dict[str, jax.Array]:
    k1, k2 = jax.random.split(key)
    return {
        "W": glorot_normal(k1, (din, dout), jnp.float32),
        "b": zeros_init(k2, (dout,), jnp.float32),
    }


def dense(p: Dict[str, jax.Array], x: jax.Array) -> jax.Array:
    return jnp.matmul(x, p["W"]) + p["b"]


def init_modified_deeponet(
    key: jax.Array,
    branch_dim: int,
    coord_dim: int,
    hidden: int,
    latent: int,
    n_layers: int,
) -> Dict[str, Any]:
    if n_layers < 2:
        raise ValueError("Modified DeepONet needs at least 2 layers.")
    if latent % 2 != 0:
        raise ValueError("Vector-output latent must be even.")

    n_dense = 2 + 2 * n_layers
    keys = iter(jax.random.split(key, n_dense))

    enc_u = init_dense(next(keys), branch_dim, hidden)
    enc_x = init_dense(next(keys), coord_dim, hidden)

    u_layers: List[Dict[str, jax.Array]] = []
    x_layers: List[Dict[str, jax.Array]] = []

    u_layers.append(init_dense(next(keys), branch_dim, hidden))
    x_layers.append(init_dense(next(keys), coord_dim, hidden))

    for _ in range(n_layers - 2):
        u_layers.append(init_dense(next(keys), hidden, hidden))
        x_layers.append(init_dense(next(keys), hidden, hidden))

    u_layers.append(init_dense(next(keys), hidden, latent))
    x_layers.append(init_dense(next(keys), hidden, latent))

    return {
        "enc_u": enc_u,
        "enc_x": enc_x,
        "u_layers": tuple(u_layers),
        "x_layers": tuple(x_layers),
    }


def init_linear_deeponet(
    key: jax.Array,
    branch_dim: int,
    coord_dim: int,
    latent: int,
) -> Dict[str, Any]:
    if latent % 2 != 0:
        raise ValueError("Vector-output linear latent must be even.")
    k1, k2 = jax.random.split(key)
    return {
        "branch": init_dense(k1, branch_dim, latent),
        "trunk": init_dense(k2, coord_dim, latent),
    }


def init_noncomposite_model(key: jax.Array) -> Dict[str, Any]:
    """
    IMPORTANT CONTROL:
    Composite code used:
        k_lf, k_lin, k_nl = jax.random.split(key, 3)
    We reproduce that exact split and discard k_lf.  Therefore the shared
    linear and nonlinear subnet initializations match the composite run.
    """
    _k_lf_unused, k_lin, k_nl = jax.random.split(key, 3)

    return {
        "linear": init_linear_deeponet(
            k_lin,
            branch_dim=2 * 41 * 41,       # exact Uc on PH=41x41
            coord_dim=2,
            latent=LINEAR_LATENT,
        ),
        "nonlinear": init_modified_deeponet(
            k_nl,
            branch_dim=(2 * 41 * 41) + (2 * 15 * 15),  # HF input + Uc on QH
            coord_dim=2,
            hidden=HIDDEN,
            latent=LATENT,
            n_layers=N_LAYERS,
        ),
    }


# =============================================================================
# PAPER MODIFIED DEEPONET FORWARD -- SAME AS COMPOSITE
# =============================================================================

def modified_forward(
    params: Dict[str, Any],
    branch: jax.Array,
    coords: jax.Array,
    activate_last: bool,
) -> jax.Array:
    """Howard modified DeepONet with vector-output latent split. Returns [B,2,P]."""
    U = jnp.tanh(dense(params["enc_u"], branch))
    V = jnp.tanh(dense(params["enc_x"], coords))

    Zu = jnp.tanh(dense(params["u_layers"][0], branch))
    Zx = jnp.tanh(dense(params["x_layers"][0], coords))

    Up = U[:, None, :]
    Vp = V[None, :, :]

    Hu = (1.0 - Zu[:, None, :]) * Up + Zu[:, None, :] * Vp
    Hx = (1.0 - Zx[None, :, :]) * Up + Zx[None, :, :] * Vp

    for pu, px in zip(params["u_layers"][1:-1], params["x_layers"][1:-1]):
        Zu = jnp.tanh(dense(pu, Hu))
        Zx = jnp.tanh(dense(px, Hx))
        Hu = (1.0 - Zu) * Up + Zu * Vp
        Hx = (1.0 - Zx) * Up + Zx * Vp

    Bu = dense(params["u_layers"][-1], Hu)
    Tx = dense(params["x_layers"][-1], Hx)

    if activate_last:
        Bu = jnp.tanh(Bu)
        Tx = jnp.tanh(Tx)

    prod = Bu * Tx
    half = prod.shape[-1] // 2
    u = jnp.sum(prod[..., :half], axis=-1)
    v = jnp.sum(prod[..., half:], axis=-1)
    return jnp.stack([u, v], axis=1)


def linear_forward(
    params: Dict[str, Any],
    branch: jax.Array,
    coords: jax.Array,
) -> jax.Array:
    """Standard one-layer no-activation DeepONet. Returns [B,2,P]."""
    b = dense(params["branch"], branch)
    t = dense(params["trunk"], coords)
    prod = b[:, None, :] * t[None, :, :]
    half = prod.shape[-1] // 2
    u = jnp.sum(prod[..., :half], axis=-1)
    v = jnp.sum(prod[..., half:], axis=-1)
    return jnp.stack([u, v], axis=1)


def branch_regularization_modified(params: Dict[str, Any]) -> jax.Array:
    """Same nonlinear branch-side weight/bias regularizer as composite code."""
    total = jnp.sum(params["enc_u"]["W"] ** 2) + jnp.sum(params["enc_u"]["b"] ** 2)
    for p in params["u_layers"]:
        total = total + jnp.sum(p["W"] ** 2) + jnp.sum(p["b"] ** 2)
    return total


# =============================================================================
# NON-COMPOSITE FORWARD -- APPENDIX D FIG. D.24
# =============================================================================

def hf_forward_noncomposite(
    params: Dict[str, Any],
    hf_branch: jax.Array,          # normalized (beta,H), [B,3362]
    uc_qh_flat: jax.Array,         # actual Uc on native 15x15, [B,450]
    uc_ph_flat: jax.Array,         # actual Uc evaluated on 41x41, [B,3362]
    coords_hf: jax.Array,          # [1681,2]
) -> Tuple[jax.Array, jax.Array, jax.Array]:
    """
    Howard Appendix D Fig. D.24 adapted to vector-valued ice velocity.

    Nonlinear branch input:
        [u(x_1..x_MH), yL(u)(x_1..x_QH)]
      -> here [normalized beta/H at HF sensors, actual Uc at QH=P_L]

    Linear branch input:
        yL(u)(x_1..x_PH)
      -> here actual Uc on the HF 41x41 output grid.
    """
    lin = linear_forward(
        params["linear"],
        uc_ph_flat,
        coords_hf,
    )

    nl_branch = jnp.concatenate([hf_branch, uc_qh_flat], axis=1)
    nl = modified_forward(
        params["nonlinear"],
        nl_branch,
        coords_hf,
        activate_last=True,
    )

    return lin + nl, lin, nl


# =============================================================================
# APPENDIX D LOSS / UPDATE
# =============================================================================

def noncomposite_loss(
    params: Dict[str, Any],
    hf_branch_batch: jax.Array,
    uc_qh_batch: jax.Array,
    uc_ph_batch: jax.Array,
    hf_target_batch: jax.Array,
    coords_hf: jax.Array,
) -> Tuple[jax.Array, Dict[str, jax.Array]]:
    """
    Howard Appendix D Eq. (D.1):
        L = lambda1 * L_HF + lambda3 * R_NL.
    """
    pred_hf, lin, nl = hf_forward_noncomposite(
        params,
        hf_branch_batch,
        uc_qh_batch,
        uc_ph_batch,
        coords_hf,
    )

    loss_hf = jnp.mean((pred_hf - hf_target_batch) ** 2)
    reg_nl = branch_regularization_modified(params["nonlinear"])

    total = LAMBDA_HF * loss_hf + LAMBDA_NL_REG * reg_nl

    aux = {
        "loss_hf": loss_hf,
        "reg_nl": reg_nl,
        "lin_rms": jnp.sqrt(jnp.mean(lin ** 2)),
        "nl_rms": jnp.sqrt(jnp.mean(nl ** 2)),
    }
    return total, aux


lr_schedule = optimizers.exponential_decay(
    LR0,
    LR_DECAY_STEPS,
    LR_DECAY_RATE,
)
opt_init, opt_update, get_params = optimizers.adam(lr_schedule)


@jit
def update_step(
    step: jax.Array,
    opt_state: Any,
    hf_branch_batch: jax.Array,
    uc_qh_batch: jax.Array,
    uc_ph_batch: jax.Array,
    hf_target_batch: jax.Array,
    coords_hf: jax.Array,
) -> Tuple[Any, jax.Array, Dict[str, jax.Array]]:
    params = get_params(opt_state)
    (loss, aux), grads = value_and_grad(noncomposite_loss, has_aux=True)(
        params,
        hf_branch_batch,
        uc_qh_batch,
        uc_ph_batch,
        hf_target_batch,
        coords_hf,
    )
    opt_state = opt_update(step, grads, opt_state)
    return opt_state, loss, aux


# =============================================================================
# EVALUATION
# =============================================================================

@jit
def predict_hf_jit(
    params: Dict[str, Any],
    hf_branch: jax.Array,
    uc_qh: jax.Array,
    uc_ph: jax.Array,
    coords_hf: jax.Array,
) -> jax.Array:
    pred, _, _ = hf_forward_noncomposite(params, hf_branch, uc_qh, uc_ph, coords_hf)
    return pred


def batched_hf_predict(
    params: Dict[str, Any],
    hf_branch: np.ndarray,
    uc_qh: np.ndarray,
    uc_ph: np.ndarray,
    coords_hf: jax.Array,
) -> np.ndarray:
    out = []
    for s in range(0, len(hf_branch), EVAL_BATCH):
        e = min(s + EVAL_BATCH, len(hf_branch))
        p = predict_hf_jit(
            params,
            jnp.asarray(hf_branch[s:e]),
            jnp.asarray(uc_qh[s:e]),
            jnp.asarray(uc_ph[s:e]),
            coords_hf,
        )
        out.append(np.asarray(jax.device_get(p), dtype=np.float32))
    return np.concatenate(out, axis=0)


# =============================================================================
# CHECKPOINT / HISTORY -- SAME STYLE AS COMPOSITE SCRIPT
# =============================================================================

def _cpu_tree(tree: Any) -> Any:
    return jax.tree_util.tree_map(lambda x: np.asarray(jax.device_get(x)), tree)


def save_checkpoint(path: Path, params: Dict[str, Any], step: int, config: Dict[str, Any]) -> None:
    payload = {
        "params": _cpu_tree(params),
        "step": int(step),
        "config": config,
    }
    with open(path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)


def save_history(path: Path, rows: List[Dict[str, float]]) -> None:
    if not rows:
        return
    keys = list(rows[0].keys())
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)


# =============================================================================
# MAIN EXPERIMENT
# =============================================================================

def run_experiment(data_path: Path, d: Dict[str, np.ndarray]) -> Dict[str, Any]:
    save_root = Path(SAVE_ROOT)
    save_root.mkdir(parents=True, exist_ok=True)

    x_lf = np.asarray(d["x_lf"], dtype=np.float32).reshape(-1)
    y_lf = np.asarray(d["y_lf"], dtype=np.float32).reshape(-1)
    x_hf = np.asarray(d["x_hf"], dtype=np.float32).reshape(-1)
    y_hf = np.asarray(d["y_hf"], dtype=np.float32).reshape(-1)

    # Same 20 paired training cases used by the ICL experiment.
    Uc_train_native = np.asarray(d["U_lf_train"][:20], dtype=np.float32)
    Uc_test_native = np.asarray(d["U_lf_test"], dtype=np.float32)

    # Appendix D linear branch needs yL at all PH output locations.
    Uc_train_hfgrid = resample_velocity_batch(
        Uc_train_native, x_lf, y_lf, x_hf, y_hf
    )
    Uc_test_hfgrid = resample_velocity_batch(
        Uc_test_native, x_lf, y_lf, x_hf, y_hf
    )

    # HF physical input and target -- identical to composite run.
    hf_branch_np = flatten_input(d["beta_hf_train"][:20], d["H_hf_train"][:20])
    hf_target_np = np.asarray(d["U_hf_train"][:20], dtype=np.float32).reshape(20, 2, 41 * 41)

    hf_test_branch_np = flatten_input(d["beta_hf_test"], d["H_hf_test"])
    hf_test_truth_np = np.asarray(d["U_hf_test"], dtype=np.float32).reshape(50, 2, 41 * 41)

    # Q_H=P_L actual LF data for nonlinear branch.
    uc_train_qh_np = Uc_train_native.reshape(20, -1).astype(np.float32)  # 450
    uc_test_qh_np = Uc_test_native.reshape(50, -1).astype(np.float32)    # 450

    # P_H actual LF data for linear branch.
    uc_train_ph_np = Uc_train_hfgrid.reshape(20, -1).astype(np.float32)  # 3362
    uc_test_ph_np = Uc_test_hfgrid.reshape(50, -1).astype(np.float32)    # 3362

    coords_hf_np = make_coords(x_hf, y_hf)
    coords_hf = jnp.asarray(coords_hf_np)

    # Strict dimensions.
    assert hf_branch_np.shape == (20, 3362)
    assert hf_target_np.shape == (20, 2, 1681)
    assert uc_train_qh_np.shape == (20, 450)
    assert uc_train_ph_np.shape == (20, 3362)
    assert hf_test_branch_np.shape == (50, 3362)
    assert uc_test_qh_np.shape == (50, 450)
    assert uc_test_ph_np.shape == (50, 3362)
    assert hf_test_truth_np.shape == (50, 2, 1681)
    assert coords_hf_np.shape == (1681, 2)

    raw_uc_stats = summarize(
        Uc_test_hfgrid.reshape(50, 2, 1681),
        hf_test_truth_np,
    )

    # Shared blocks get the same subkeys they got in the composite model.
    key = jax.random.PRNGKey(SEED)
    params0 = init_noncomposite_model(key)
    opt_state = opt_init(params0)

    leaves = jax.tree_util.tree_leaves(params0)
    n_params = int(sum(np.prod(x.shape) for x in leaves))

    print("\n" + "=" * 118)
    print("HOWARD APPENDIX-D NON-COMPOSITE MF | VERSION A | GP SCALE s=0.35")
    print("=" * 118)
    print("Dataset                    :", data_path)
    print("HF train / test            : 20 / 50")
    print("Actual paired Uc           : YES (LF train[0:20], all 50 LF test)")
    print("Model                      : linear 1x10 + nonlinear modified 5x200")
    print("Removed                    : learned LF modified DeepONet")
    print("Nonlinear input            : 3812 = 3362 HF(beta,H) + 450 actual Uc@QH")
    print("Linear input               : 3362 actual Uc@PH")
    print("Q_H                        : P_L = 450")
    print("Model seed                 :", SEED)
    print("Shared-subnet init         : SAME k_lin/k_nl subkeys as composite")
    print("HF minibatch sequence      : SAME RNG sequence as composite")
    print("HF batch                   :", HF_BATCH)
    print("Optimizer                  : JAX Adam")
    print("LR schedule                : (1e-4, 5000, 0.97)")
    print("Training steps             :", TRAIN_STEPS)
    print("Loss                       : lambda_HF*L_HF + lambda_NLreg*R_NL")
    print("Lambda source              :", LAMBDA_SOURCE)
    print("lambda_HF / lambda_NLreg   :", LAMBDA_HF, "/", LAMBDA_NL_REG)
    print("Output scaling             : NONE; physical m/yr")
    print("Input nondim               : beta/3000, H/1000, coords -> [-1,1]")
    print("Parameters                 :", f"{n_params:,}")
    print("Official checkpoint        : FINAL fixed-budget step only")
    print("=" * 118)

    print("\n[tensor audit]")
    print("  HF branch/target         :", hf_branch_np.shape, hf_target_np.shape)
    print("  actual Uc @ Q_H          :", uc_train_qh_np.shape)
    print("  actual Uc @ P_H          :", uc_train_ph_np.shape)
    print("  test HF branch/truth     :", hf_test_branch_np.shape, hf_test_truth_np.shape)
    print("  test actual Uc Q_H/P_H   :", uc_test_qh_np.shape, uc_test_ph_np.shape)
    print("\n[raw paired Uc -> HF test baseline]")
    print(" ", fmt_stats(raw_uc_stats))

    config = {
        "paper": "Howard et al., JCP 493 (2023) 112462",
        "framework": "Appendix D non-composite, adapted to Sec. 3.4.2 ice-sheet benchmark",
        "dataset": str(data_path),
        "dataset_sha256": sha256_file(data_path),
        "dataset_gp_scale": 0.35,
        "N_H": 20,
        "N_test": 50,
        "paired_Uc_source": "U_lf_train[0:20] and U_lf_test from same Version-A benchmark",
        "M_H": 3362,
        "P_H": 3362,
        "Q_H": 450,
        "linear_layers": 1,
        "linear_latent": 10,
        "nonlinear_layers": 5,
        "nonlinear_hidden": 200,
        "main_latent": 200,
        "activation": "tanh",
        "nonlinear_activate_last": True,
        "linear_activation": "none",
        "optimizer": "JAX adam",
        "lr0": LR0,
        "decay_steps": LR_DECAY_STEPS,
        "decay_rate": LR_DECAY_RATE,
        "steps": TRAIN_STEPS,
        "hf_batch": HF_BATCH,
        "lambda_source": LAMBDA_SOURCE,
        "lambda_hf": LAMBDA_HF,
        "lambda_nl_reg": LAMBDA_NL_REG,
        "loss": "Appendix D Eq.(D.1): lambda1*L_HF + lambda3*R_NL",
        "regularization_scope": "nonlinear branch-side encoder+branch layers only; same as composite implementation",
        "linear_regularization": "none",
        "lf_network": "none; actual Uc supplied directly",
        "uc_qh": "actual native 15x15 Uc, flattened to 450",
        "uc_ph": "same actual Uc bilinearly interpolated 15x15->41x41 on physical grid",
        "shared_initialization_control": "discard composite k_lf subkey; use identical k_lin and k_nl subkeys",
        "hf_minibatch_control": "consume same unused LF RNG draw before each HF draw to match composite HF minibatch sequence",
        "vector_output": "first half latent=u, second half=v",
        "output_scaling": "none; physical m/yr",
        "input_nondim": {
            "beta_divisor": BETA_SCALE,
            "H_divisor": H_SCALE,
            "coords": "mapped independently to [-1,1]",
        },
        "checkpoint_selection": "final fixed-budget step only; test evaluated only after training",
        "important_scope_note": (
            "Howard Appendix D defines non-composite MF but does not report this ice-sheet experiment; "
            "this run combines Appendix-D architecture/loss with Sec.3.4.2 ice-sheet sizes/hyperparameters."
        ),
    }

    with open(save_root / "config.json", "w") as f:
        json.dump(config, f, indent=2)

    rng = make_rng(SEED)
    history: List[Dict[str, float]] = []

    print("\n[train] Starting non-composite training...")
    t0 = time.time()
    last_loss = math.nan
    last_aux: Dict[str, float] = {}

    for step in range(TRAIN_STEPS):
        # Deliberately reproduce the composite code's RNG consumption so the
        # following HF minibatch indices are the same sequence.
        _unused_il = rng.choice(100, size=LF_BATCH_FOR_RNG_MATCH, replace=False)
        ih = rng.choice(20, size=HF_BATCH, replace=False)

        opt_state, loss, aux = update_step(
            jnp.asarray(step, dtype=jnp.int32),
            opt_state,
            jnp.asarray(hf_branch_np[ih]),
            jnp.asarray(uc_train_qh_np[ih]),
            jnp.asarray(uc_train_ph_np[ih]),
            jnp.asarray(hf_target_np[ih]),
            coords_hf,
        )

        human_step = step + 1

        if human_step == 1 or human_step % LOG_EVERY == 0 or human_step == TRAIN_STEPS:
            loss_h = float(jax.device_get(aux["loss_hf"]))
            reg_nl = float(jax.device_get(aux["reg_nl"]))
            lin_rms = float(jax.device_get(aux["lin_rms"]))
            nl_rms = float(jax.device_get(aux["nl_rms"]))
            last_loss = float(jax.device_get(loss))

            weighted_h = LAMBDA_HF * loss_h
            weighted_nlreg = LAMBDA_NL_REG * reg_nl
            lr_now = float(lr_schedule(step))

            print(
                f"step {human_step:06d}/{TRAIN_STEPS} | "
                f"total={last_loss:.6e} | "
                f"HF={loss_h:.6e} | regNL={reg_nl:.3e} | "
                f"wHF={weighted_h:.3e} | wNLreg={weighted_nlreg:.3e} | "
                f"linRMS={lin_rms:.3e} | nlRMS={nl_rms:.3e} | "
                f"lr={lr_now:.3e} | elapsed={(time.time()-t0)/60.0:.1f}m"
            )

            last_aux = {
                "loss_hf": loss_h,
                "reg_nl": reg_nl,
                "lin_rms": lin_rms,
                "nl_rms": nl_rms,
            }

        # Held-out test is intentionally NOT evaluated during training.

        if human_step % CHECKPOINT_EVERY == 0 or human_step == TRAIN_STEPS:
            save_checkpoint(
                save_root / f"step{human_step:06d}.pkl",
                get_params(opt_state),
                human_step,
                config,
            )
            print(f"  [checkpoint] saved step {human_step}")

    params_final = get_params(opt_state)
    pred_test = batched_hf_predict(
        params_final,
        hf_test_branch_np,
        uc_test_qh_np,
        uc_test_ph_np,
        coords_hf,
    )
    final = summarize(pred_test, hf_test_truth_np)

    print("\n" + "=" * 118)
    print("OFFICIAL FINAL NON-COMPOSITE MF | LAYER-1 CANONICAL s=0.35")
    print(fmt_stats(final))
    print("=" * 118)
    print("Raw paired Uc baseline:", fmt_stats(raw_uc_stats))

    save_checkpoint(save_root / "final.pkl", params_final, TRAIN_STEPS, config)

    rel = per_case_rel_l2(pred_test, hf_test_truth_np)
    np.savez_compressed(
        save_root / "test_predictions.npz",
        prediction=pred_test.astype(np.float32),
        target=hf_test_truth_np.astype(np.float32),
        rel_l2=rel.astype(np.float64),
        actual_Uc_hfgrid=Uc_test_hfgrid.astype(np.float32),
    )

    summary = {
        "final_test": final,
        "raw_uc_test": raw_uc_stats,
        "elapsed_min": (time.time() - t0) / 60.0,
        "config": config,
    }
    with open(save_root / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    return summary


def main() -> None:
    print("=" * 118)
    print("LAYER-1 FINAL | NON-COMPOSITE MF | CANONICAL s=0.35")
    print("=" * 118)
    print("JAX version :", jax.__version__)
    print("JAX devices :", jax.devices())
    print("Input root  :", INPUT_ROOT)
    print("Dataset file:", VERSION_A_FILENAME)
    print("Lambda mode :", LAMBDA_SOURCE)
    print("=" * 118)

    data_path = resolve_data_path(VERSION_A_FILENAME)
    d = load_version_a(data_path)
    meta = audit_canonical_metadata(d)
    digest = sha256_file(data_path)
    print("[canonical audit] beta_gp_scale = 0.35")
    print("[canonical audit] SHA256        =", digest)

    # Metadata audit if present.
    if "metadata_json" in d:
        try:
            raw = d["metadata_json"]
            if isinstance(raw, np.ndarray):
                raw = raw.item()
            meta = json.loads(str(raw))
            cfg = meta.get("config", meta)
            if "beta_gp_scale" in cfg:
                print("[dataset audit] beta_gp_scale =", cfg["beta_gp_scale"])
        except Exception:
            pass

    summary = run_experiment(data_path, d)

    root = Path(SAVE_ROOT)
    root.mkdir(parents=True, exist_ok=True)
    with open(root / f"single_run_summary_{LAMBDA_SOURCE}.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\nDONE")


if __name__ == "__main__":
    main()


LAYER-1 FINAL | NON-COMPOSITE MF | CANONICAL s=0.35
JAX version : 0.7.2
JAX devices : [CudaDevice(id=0), CudaDevice(id=1)]
Input root  : /kaggle/input
Dataset file: howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz
Lambda mode : layer1_final_table_c7
[data-path] howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz
  -> /kaggle/input/notebooks/xli46913/canonical-dataset/LAYER1_CANONICAL_gpScale0p35/howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz
[canonical audit] beta_gp_scale = 0.35
[canonical audit] SHA256        = 6786500bb1f34939ea3a589cbe5ff1bf23b995eb9a3580cd6cfb11465dfe4875
[dataset audit] beta_gp_scale = 0.35

HOWARD APPENDIX-D NON-COMPOSITE MF | VERSION A | GP SCALE s=0.35
Dataset                    : /kaggle/input/notebooks/xli46913/canonical-dataset/LAYER1_CANONICAL_gpScale0p35/howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz
HF train / test            : 20 / 50
Actual paired Uc           : YES (LF train[0:20], all 50 LF test)
Model        